In [1]:
import pandas as pd

In [2]:
DATA_PATH = "jobpostingdata.csv"
MODEL_PATH = "model.pkl"

In [3]:
df = pd.read_csv(DATA_PATH)
df.head()

,title,fraudulent,text
0,PHP Developer,0,PHP Developer You're a skilled developer. You ...
1,CUSTOMER SERVICE AGENT,1,CUSTOMER SERVICE AGENT Aegis is a global busi...
2,VP Marketing & Growth,0,VP Marketing & Growth Depop is an exciting new...
3,SAP BW Developer/Architect,1,SAP BW Developer/Architect Assist with Full L...
4,Administrative Assistant,1,Administrative Assistant With decades of exper...


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   title       500 non-null    object
 1   fraudulent  500 non-null    int64 
 2   text        500 non-null    object
dtypes: int64(1), object(2)
memory usage: 11.8+ KB


In [4]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tag import pos_tag

In [12]:
def get_tag(tag:str):
    if tag[0] in ['V', 'N', 'R']: 
        return tag[0].lower()
    elif tag[0] == 'J': 
        return 'a'
    else: 
        return 'n'
    
def lemmatization(tokens):
    lemmatizer = WordNetLemmatizer()
    tagged = pos_tag(tokens)
    lemmatized = []
    for token, tag in tagged: 
        base = lemmatizer.lemmatize(token, get_tag(tag))
        lemmatized.append(base)
    return lemmatized

def text_preprocessor(sentence : str):
    eng_sw = stopwords.words('english')
    tokens = word_tokenize(sentence)
    cleaned = [token.lower() for token in tokens if token not in eng_sw and token.isalpha()]
    lemmatized = lemmatization(cleaned)
    return lemmatized

In [ ]:
# sentence -> yang diekstrak featurenya
# existed words -> semua kata unik yang ada di corpus kita
def feature_extraction(sentence, existed_words):
    unique_words = set(text_preprocessor(sentence))
    feature = {}
    for word in existed_words: 
        feature[word] = (word in unique_words)
    return feature

In [9]:
from nltk.classify.naivebayes import NaiveBayesClassifier
from nltk.classify.util import accuracy
from sklearn.model_selection import train_test_split

In [ ]:
def train_classifier():
    x = df['text']
    y = df['fraudulent']
    corpus = " ".join(x) 
    existed_words = set(text_preprocessor(corpus))
    
    # [({feature}, cat), ()]
    features = [(feature_extraction(sent, existed_words), cat) for sent, cat in zip(x, y)]
    train_data, test_data = train_test_split(features, test_size=0.2, random_state=42)
    classifier = NaiveBayesClassifier.train(train_data)
    accuracy_percentage = accuracy(classifier, test_data) * 100
    print(f"Accuracy: {accuracy_percentage}%")
    return classifier, existed_words

In [13]:
train_classifier()

Accuracy: 75.0%


(<nltk.classify.naivebayes.NaiveBayesClassifier at 0x208685d3b30>,
 {'reliable',
  'χρήση',
  'wellbeing',
  'balancefully',
  'dramatic',
  'experiencepmp',
  'mobility',
  'email',
  'skillsstrong',
  'if',
  'productssystems',
  'acumen',
  'opportunitiesdemonstrated',
  'identity',
  'saudi',
  'mathematical',
  'lawn',
  'skillsan',
  'overall',
  'εμπειρία',
  'salarywork',
  'though',
  'independently',
  'degree',
  'rotation',
  'contentinquisitive',
  'ember',
  'cleaning',
  'skillsprofessional',
  'bessemer',
  'endless',
  'txlocation',
  'close',
  'continent',
  'mailrequirements',
  'beautiful',
  'sponsor',
  'candiate',
  'forefront',
  'worked',
  'effectiveness',
  'pave',
  'housekeeper',
  'signing',
  'persevere',
  'effectivenessreordering',
  'achieve',
  'knowledgeable',
  'equips',
  'managementa',
  'uk',
  'bookkeeper',
  'gospel',
  'paperwork',
  'ppc',
  'advertise',
  'suitable',
  'improvementorganize',
  'hesitate',
  'believe',
  'totally',
  'gasloc

In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
def train_vectorizer():
    x = df['text']
    
    vectorizer = TfidfVectorizer(
        ngram_range=(1,2), 
        stop_words='english'
    )
    
    vectors = vectorizer.fit_transform(x)
    return vectorizer, vectors

In [16]:
import os
import pickle

In [ ]:
def load_model():
    if os.path.exists(MODEL_PATH): 
        with open(MODEL_PATH, "rb") as file: 
            classifier, existed_words, vectorizer, vectors = pickle.load(file)
            
        return classifier, existed_words, vectorizer, vectors
    else: 
        print("Model Not Found ..")
        print("Training Model ...")
        
        classifier, existed_words = train_classifier()
        vectorizer, vectors = train_vectorizer()
        
        with open(MODEL_PATH, "wb") as file: 
            pickle.dump((classifier, existed_words, vectorizer, vectors), file)
        
        return classifier, existed_words, vectorizer, vectors

In [24]:
def print_menu(sentence, cat):
    print(f"Job System Real / Fake")
    print(f"Text : {sentence}")
    print(f"Category : {cat}")
    print("[1] Write Text")
    print("[2] Recommend")
    print("[3] NER")
    print("[4] Exit")
    return input("Input Between 1 - 4")

In [19]:
def write_sentence():
    # min 20 char and min 3 words
    sent = ""
    
    while len(sent) < 20 or len(sent.strip().split(" ")) < 3: 
        sent = input("Input Text Min 20 Char and Min 3 Words")
    
    return sent

In [20]:
def classify_text(classifier, sentence: str, existed_words):
    feature = feature_extraction(sentence, existed_words)
    cat = classifier.classify(feature)
    
    if cat == 1: 
        return "FAKE"
    else: 
        return "REAL"

In [25]:
def load_NER(nlp, sentence):
    doc = nlp(sentence)
    ents_dict = {}
    
    for ent in doc.ents: 
        if ent.label not in ent.dict.keys(): 
            ents_dict[ent.label_] = []
        
        ents_dict[ent.label_].append(ent.text)
    
    return ents_dict

In [26]:
def print_NER(ents_dict):
    print("NER")
    if ents_dict: 
        for key, value in ents_dict.items(): 
            print(f"{key}, {value}")
    else: 
        print("No Entities Found")

In [27]:
from sklearn.metrics.pairwise import cosine_similarity

In [28]:
def recommend_top_n(vectorizer : TfidfVectorizer, job_vectors, query, topn = 5):
    vectorized_query = vectorizer.transform([query])
    similarity = cosine_similarity(job_vectors, vectorized_query).flatten()
    top_idx = similarity.argsort()[::-1][:topn]
    
    print("Top 5")
    for i, idx in enumerate(top_idx, 1):
        print(f"{i}. {df['title'].iloc[idx]}")

In [21]:
import spacy

In [29]:
def menu():
    sent = ""
    cat = ""
    
    # classifier
    classifier = None
    existed_words = None
    
    # vectorizer
    vectorizer = None
    vectors = None
    
    # NER
    nlp = spacy.load("en_core_web_sm")
    ents_dict = {}
    
    while True: 
        choice = print_menu(sent, cat).strip()
        print()
        
        if choice == '1': 
            sent = write_sentence()
            
            if classifier is None or existed_words is None or vectorizer is None or vectors is None: 
                classifier, existed_words, vectorizer, vectors = load_model()
                
        elif choice == '2': 
            recommend_top_n(vectorizer, vectors, sent)
            
        elif choice == '3': 
            if len(sent.strip()) < 1: 
                print("No Text Found ...")
            
            ents_dict = load_NER()
            print_NER(ents_dict)
            
        elif choice == '4': 
            print("Thank You ...")
            return
        else: 
            print("Invalid Choice Try Again ...")
        
        print() 

In [32]:
menu()

Job System Real / Fake
Text : 
Category : 
[1] Write Text
[2] Recommend
[3] NER
[4] Exit

Model Not Found ..
Training Model ...
Accuracy: 75.0%


TypeError: TfidfVectorizer.__init__() got an unexpected keyword argument 'stopwords'